In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

images=os.listdir("C:\\Users\\Himanshi\\Desktop\\SDP\\DataSet\\skin-disease-datasaet\\train_set\\BA- cellulitis")
print(type(images))
for img in images:
    img_r=cv2.imread(os.path.join("C:\\Users\\Himanshi\\Desktop\\SDP\\DataSet\\skin-disease-datasaet\\train_set\\BA- cellulitis",img))
    plt.imshow(cv2.cvtColor(img_r, cv2.COLOR_BGR2RGB))    
    plt.figure()
    plt.imshow(img_r)
plt.close()

In [ ]:
from keras.preprocessing.image import ImageDataGenerator
train_data_dir = 'C:\\Users\\Himanshi\\Desktop\\SDP\\DataSet\\skin-disease-datasaet\\train_set'
test_data_dir = 'C:\\Users\\Himanshi\\Desktop\\SDP\\DataSet\\skin-disease-datasaet\\test_set'
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalize pixel values
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,  # Apply data augmentation
)
test_datagen = ImageDataGenerator(rescale=1./255) 
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(256, 256),  # Adjust target size as needed
    batch_size=32,  # Experiment with batch size for optimal speed
    class_mode='categorical',
)
test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical',
)

In [ ]:
from keras.applications import VGG16
from keras.models import Model
from keras.layers import Dense, Flatten
from keras.optimizers import Adam

# Load the pre-trained VGG16 model without the top (fully connected) layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(256, 256, 3))

# Freeze the weights of the pre-trained layers so they are not updated during training
for layer in base_model.layers:
    layer.trainable = False

# Add custom fully connected layers on top of the base model
num_classes= 8
x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
output = Dense(num_classes, activation='softmax')(x)  # Adjust num_classes according to your dataset

# Create the final model
model = Model(inputs=base_model.input, outputs=output)

# Compile the model
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

# Train the model using the data generators
history = model.fit_generator(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=10,  # Adjust number of epochs as needed
    validation_data=test_generator,
    validation_steps=len(test_generator)
)

# Evaluate the model on the test data
test_loss, test_accuracy = model.evaluate_generator(test_generator, steps=len(test_generator))
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)